# **SAD2 lab 1**

*Today's lab is based on the lab scenario by Kazik Oksza-Orzechowski.*


# Introduction to PyTorch

In today's lab, we will familiarize ourselves with [PyTorch](https://pytorch.org/docs/stable/index.html). At the beginning of the semester we will mainly use PyTorch [Distributions](https://pytorch.org/docs/stable/distributions.html), however being familiar with PyTorch will be important later in the semester for the classes about Variational Autoencoders.

First let's follow quick tutorial based on [this source](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial2/Introduction_to_PyTorch.html).

## Basics

We will start with reviewing the basic concepts of PyTorch. As a prerequisite, we recommend to be familiar with the `numpy` package as most machine learning frameworks are based on very similar concepts. If you are not familiar with `numpy` yet, don't worry: [here](https://numpy.org/doc/stable/user/quickstart.html) is a tutorial to go through.

Let's start with importing PyTorch. The package is called `torch`, based on its original framework [Torch](http://torch.ch/). As a first step, we can check its version:

In [ ]:
import torch

print("Using torch", torch.__version__)

As in every machine learning framework, PyTorch provides functions that are stochastic like generating random numbers. However, a very good practice is to setup your code to be reproducible with the exact same random numbers. This is why we set a seed below. As everyone knows, 42 is the [best seed](https://medium.com/geekculture/the-story-behind-random-seed-42-in-machine-learning-b838c4ac290a).

In [ ]:
torch.manual_seed(42) # Setting the seed

For custom operators, you might need to set python seed as well:


In [ ]:
import random
random.seed(42)

If you or any of the libraries you are using rely on NumPy, you can seed the global NumPy RNG with:

In [ ]:
import numpy as np
np.random.seed(42)

## Tensors



Let’s first start by looking at different ways of creating a tensor. There are many possible options, the simplest one is to call `torch.Tensor` passing the desired shape as input argument:

In [ ]:
x = torch.Tensor(2, 3, 4)
print(x)

The function `torch.Tensor` allocates memory for the desired tensor, but reuses any values that have already been in the memory. To directly assign values to the tensor during initialization, there are many alternatives including:

* `torch.zeros`: Creates a tensor filled with zeros
* `torch.ones`: Creates a tensor filled with ones
* `torch.rand`: Creates a tensor with random values uniformly sampled between 0 and 1
* `torch.randn`: Creates a tensor with random values sampled from a normal distribution with mean 0 and variance 1
* `torch.arange`: Creates a tensor containing the values $N,N+1,N+2,...,M$
* `torch.Tensor` (input list): Creates a tensor from the list elements you provide

In [ ]:
# Create a tensor from a (nested) list
x = torch.Tensor([[1, 2], [3, 4]])
print(x)

In [ ]:
# Create a tensor with random values between 0 and 1 with the shape [2, 3, 4]
x = torch.rand(2, 3, 4)
print(x)

You can obtain the shape of a tensor in the same way as in `numpy` (`x.shape`), or using the `.size` method:

In [ ]:
shape = x.shape
print("Shape:", x.shape)

size = x.size()
print("Size:", size)

dim1, dim2, dim3 = x.size()
print("Size:", dim1, dim2, dim3)

## Tensors and NumPy

Tensors can be converted to numpy arrays, and numpy arrays back to tensors. To transform a numpy array into a tensor, we can use the function `torch.from_numpy`:

In [ ]:
import numpy as np

np_arr = np.array([[1, 2], [3, 4]])
tensor = torch.from_numpy(np_arr)

print("Numpy array:", np_arr)
print("PyTorch tensor:", tensor)

To transform a PyTorch tensor back to a numpy array, we can use the function `.numpy()` on tensors:

In [ ]:
tensor = torch.arange(4)
np_arr = tensor.numpy()

print("PyTorch tensor:", tensor)
print("Numpy array:", np_arr)

The conversion of tensors to numpy require the tensor to be on the CPU, and not the GPU (more on GPU support in a later section). In case you have a tensor on GPU, you need to call `.cpu()` on the tensor beforehand. Hence, you get a line like `np_arr = tensor.cpu().numpy()`.

## Operations on tensors

Most operations that exist in numpy, also exist in PyTorch. A full list of operations can be found in the [PyTorch documentation](https://pytorch.org/docs/stable/tensors.html#), but we will review the most important ones here.

The simplest operation is to add two tensors:

In [ ]:
x1 = torch.rand(2, 3)
x2 = torch.rand(2, 3)
y = x1 + x2

print("X1", x1)
print("X2", x2)
print("Y", y)

Calling `x1 + x2` creates a new tensor containing the sum of the two inputs. However, we can also use in-place operations that are applied directly on the memory of a tensor. We therefore change the values of `x2` without the chance to re-accessing the values of `x2` before the operation. An example is shown below:

In [ ]:
x1 = torch.rand(2, 3)
x2 = torch.rand(2, 3)
print("X1 (before)", x1)
print("X2 (before)", x2)

x2.add_(x1)
print("X1 (after)", x1)
print("X2 (after)", x2)

In-place operations are usually marked with a underscore postfix (e.g. `add_` instead of `add`).

Another common operation aims at changing the shape of a tensor. A tensor of size $(2,3)$ can be re-organized to any other shape with the same number of elements (e.g. a tensor of size $(6)$, or $(3,2)$, etc.). In PyTorch, this operation is called `view`:

In [ ]:
x = x1.view(2, 3)
print("X", x)

Other commonly used operations include matrix multiplications, which are essential for neural networks. Quite often, we have an input vector $\mathbf{x}$, which is transformed using a learned weight matrix $\mathbf{W}$. There are multiple ways and functions to perform matrix multiplication, some of which we list below:

* `torch.matmul`: Performs the matrix product over two tensors, where the specific behavior depends on the dimensions. If both inputs are matrices (2-dimensional tensors), it performs the standard matrix product. For higher dimensional inputs, the function supports broadcasting (for details see the [documentation](https://pytorch.org/docs/stable/generated/torch.matmul.html?highlight=matmul#torch.matmul)). Can also be written as `a @ b`, similar to numpy.
* `torch.mm`: Performs the matrix product over two matrices, but doesn't support broadcasting (see [documentation](https://pytorch.org/docs/stable/generated/torch.mm.html?highlight=torch%20mm#torch.mm))
* `torch.bmm`: Performs the matrix product with a support batch dimension. If the first tensor $T$ is of shape ($b\times n\times m$), and the second tensor $R$ ($b\times m\times p$), the output $O$ is of shape ($b\times n\times p$), and has been calculated by performing $b$ matrix multiplications of the submatrices of $T$ and $R$: $O_i = T_i @ R_i$
* `torch.einsum`: Performs matrix multiplications and more (i.e. sums of products) using the Einstein summation convention. Explanation of the Einstein sum can be found in assignment 1.

Usually, we use `torch.matmul` or `torch.bmm`. We can try a matrix multiplication with `torch.matmul` below.

In [ ]:
x1 = torch.rand(2, 3)
x2 = torch.rand(3, 2)
print("X1", x1)
print("X2", x2)

y = x1.matmul(x2)

print("Y", y)

## Indexing

We often have the situation where we need to select a part of a tensor. Indexing works just like in numpy, so let's try it:

In [ ]:
x = torch.arange(12).view(3, 4)
print("X: ", x)

print("Second column")
print(x[:, 1])

print("First row")
print(x[0])

print("First two rows, last column")
print(x[:2, -1])

print("Middle two rows")
print(x[1:3, :])

# PyTorch Distributions

Source: https://pytorch.org/docs/stable/distributions.html

The `distributions` package contains parameterizable probability distributions and sampling functions. This allows the construction of stochastic computation graphs and stochastic gradient estimators for optimization. This package generally follows the design of the [TensorFlow Distributions](https://arxiv.org/abs/1711.10604) package.

During this course, we focus on probabilistic modelling and will be working with various probabilistic distribution. Thus we need to be able to:

- efficiently sample from the distribution,

- efficiently compute pointwise probability density,

- differentiate probability density funtion w.r.t the parameters.

Those conditions generally met by the distributions available in PyTorch Distributions.


## Exercise 1: Sampling from Multivariate Normal distribution

1. Import `torch`, print the version you are using.

2. Set a seed, you can use `set_seed` function below.

3. Define a multivariate normal distribution with $\mu = [1, 2]$ and $\Sigma = \begin{bmatrix} 1 & 0 \\ 0 & 2\end{bmatrix}$. Use `torch.distributions.MultivariateNormal`.

4. SSample $n = 100$ i.i.d. observations from the defined distribution using `sample` method (Hint: `sample_shape = torch.Size([n])`).

5. Compute the probability and log-probability of the sampled batch obtained in 4 under the distribution from 3 (see: `log_prob` method).

6. Explain **why do we often work with log-probs not probs?**

7. How is log-probability of density function  connected to the likelihood? What is the difference?

In [ ]:
# 1, 2
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    print(f"Seed set to {seed}")
set_seed(1918)

In [ ]:
# 3

# 4 

# 5 

### Exercise 1.6
First we need to justify this transformation
- log function is strictly increasing, that means we do not change behavior of function (extremum will be the same):
$$\frac{\mathrm{d}}{\mathrm{d}x}f(g(x)) = \frac{\mathrm{d}f}{\mathrm{d}x}(g(x)) \cdot \frac{\mathrm{d}g(x)}{\mathrm{d}x}$$
Where:
- $f$ is strictly increasing

Why is it useful? 
- [exponential families](https://en.wikipedia.org/wiki/Exponential_family): it allows ous to 
Often we will deal with  and also we will be dealing 

In [ ]:
############################################################
# Put your code here

############################################################

Shapes of distributions may be confusing. A perfect guide to make your life easier: https://bochang.me/blog/posts/pytorch-distributions/

Additionally: https://stats.stackexchange.com/questions/2641/what-is-the-difference-between-likelihood-and-probability


## Likelihood of Coin Tossing

### Coin tossing

The coin tossing experiment can be described using Bernoulli distribution, where $\theta$ is the probability of heads: $\mathbb{P}(X = 1) = \theta$ and $1 - \theta$ is probability of tails: $\mathbb{P}(x = 0) = 1 - \theta$.

The probability mass function of Bernoulli random variable $X$ can be written as

$$f(X) = \theta^X(1 - \theta)^{1 - X}.$$

PMF written this way is clearly derivable.

### Likelihood function

**Likelihood function** describes the probability of the observed data given the model. Let $\mathcal{D}$ be observed data and $\theta$ be parameter of the model, then the likelihood is a function of the parameter $\theta$ and can be expressed as $\mathbb{P}(\mathcal{D} | \theta)$.

Now, let $\mathcal{D} = \{X_1, \dots, X_N\}$ be $N$ observed data points from coin tossing experiment ($X_i$ - i.i.d. random variables). Then **Likelihood function** can be expressed as:

$$L(\theta) = \mathbb{P}(\mathcal{D} | \theta) = \prod_{i=1}^N \theta^{X_i} (1 - \theta)^{1 - X_i}.$$

We often work with log likelihood functions since they are easier to compute. We have:

$$\begin{align*}
LL(\theta) &= \log \Bigl( \prod_{i=1}^N \theta^{X_i} (1 - \theta)^{1 - X_i} \Bigl) \\
&= \sum_{i=1}^{N} \log \Bigl( \theta^{X_i}(1 - \theta)^{1-X_i} \Bigl) \\
& = \sum_{i=1}^{N} X_i \log(\theta) + (1 - X_i) \log(1 -\theta) \\
&= Y\log(\theta) + (N - Y) \log( 1 -\theta),
\end{align*}$$

where $Y = \sum_{i=1}^{N} X_i$.

#MLE

Since likelihood and log-likelihood functions have the same maximum, we are going to find MLL, which is $\hat{\theta}$ that maximizes $LL(\theta)$.

$$LL'(\theta) = Y\frac{1}{\hat{\theta}} + (N-Y)\frac{-1}{1-\hat{\theta}} $$

$$Y(\frac{1}{\hat{\theta}} - 1) - N + Y = 0 $$

$$\hat{\theta} = \frac{Y}{N} = \frac{\sum_{i=1}^{N} X_i}{N}$$

Since $LL(\theta)$ is concave, then this extremum is the maximum.

MLE for Bernoulli distribution is simply the sample mean.


## Exercise 2

1. Write function `simulate_coin_tosses` that simulates $n$ coin tosses, with probability of heads equal $\theta$. Use `torch.distributions.Bernoulli` (Hint: use `expand` method to change tensor shape).

2. Simulate $n = 4$ unfair coin tosses, with a probability of heads equal to $\theta = 0.6$.

3. Write a function `get_bernoulli_loglikelihood_analytical` that calculates the log-likelihood of the generated data using the analytical formula (above). Use `torch.log()` function.

4. Write function `get_bernoulli_likelihood_analytical` either using the analytical formula from above or using previous function.

5. Write the function `get_bernoulli_loglikelihood` that calculates the log-likelihood of the generated data using the `torch.distributions`.  White function `get_bernoulli_likelihood` using previous funtion (HINT: likelihood = exp(loglikelihood)).

6. For 101 different, equally spaced values of $\theta$ (Hint: use `torch.arrange()` function, include 0 and 1, since  $\theta \in [0, 1]$) compute **log-likelihood** and **likelihood** using both functions. Verify whether the outcomes are close.

7. Plot the **likelihood** function for 101 different, equally spaced values of $\theta$.

8. Mark the **maximum likelihood estimate (MLE)** on the plot.

9. Repeat the experiment $100$ times, is the MLE consistent across iterations? How do we call this phenomenon?

10. Repeat all the above steps for $N = 1000$. What happened? What is the minimum and maximum value for MLE across iterations?

In [ ]:
# Helper functions

def simulate_coin_tosses(n, theta):
  pass

def get_bernoulli_loglikelihood_analytical(data, theta):
  pass

def get_bernoulli_likelihood_analytical(data, theta):
  pass

def get_bernoulli_loglikelihood(data, theta):
  pass

def get_bernoulli_likelihood(data, theta):
  pass

def get_bernoulli_mle(data):
  pass

In [ ]:
N = 4
THETA = 0.6

biased_coin_tosses = simulate_coin_tosses(n = N, theta = THETA)

###################################################################
# put your code here
theta_values = ...
###################################################################

# Compute log-likelihoods
likelihoods_analytical = torch.stack(
    [get_bernoulli_likelihood_analytical(biased_coin_tosses, theta) for theta in theta_values]
)

likelihoods = torch.stack(
    [get_bernoulli_likelihood(biased_coin_tosses, theta) for theta in theta_values]
)

if not torch.equal(likelihoods_analytical, likelihoods):
  differences = torch.abs(likelihoods_analytical - likelihoods)
  print(f"Max difference: {differences.max()}.")
  print(f"Mean difference: {differences.mean()}")

# You may use a variant of torch.equal()
print(torch.allclose(likelihoods, likelihoods_analytical))

import matplotlib.pyplot as plt
import seaborn as sns
ax = sns.lineplot(x=theta_values, y=likelihoods)
ax.set(xlabel="likelihood parameter", ylabel="likelihood", title=f"points")
plt.axvline(get_bernoulli_mle(biased_coin_tosses))

plt.show()

In [ ]:
n_iter = 100
mle_values = torch.zeros(n_iter)
for iter in range(n_iter):
  biased_coin_tosses = simulate_coin_tosses(n = N, theta = THETA)
  mle_values[iter] = get_bernoulli_mle(biased_coin_tosses)
print("Minimum value of MLE: ", mle_values.min())
print("Maximum value of MLE: ", mle_values.max())